# Exercise 2b: The same data without labels – anomaly detection
**Internet of Production – AI, week 1** · about 20 min · work in pairs

**Thread:** In exercise 2 you trained a classifier with **labels** ("blockage within 6 hours"). In practice such labels are rare: a new printer has no failure history, and failures are few. Here you use **only the normal behaviour** of the printer and report everything that **deviates** from it.

**Task:** Learn what is normal from the first 70 % of the data, without using the label `y`. Raise an alarm when the energy of the last two hours is unusually high. The label is only used to **evaluate** the alarms.

Every `___` is a gap.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

DATA = Path('../../00_Data/printfarm')
HORIZON = 360                       # minutes
FEATURES = ['E30', 'E120', 'slope']

def make_blockage_data(k, horizon=HORIZON):
    """The pipeline of exercise 2 in one function: one row per 10th printing minute."""
    d = pd.read_csv(DATA / f'Printer_{k}.csv', sep=';', decimal=',', low_memory=False).dropna(how='all')
    d['Timestamp'] = pd.to_datetime(d['Timestamp'], format='%d.%m.%Y %H:%M')
    d = d.set_index('Timestamp')
    d['Energy'] = d['Total_Power'].diff()
    e = pd.read_csv(DATA / f'Printer_{k}_error.csv', sep=';').dropna(how='all')
    e['Timestamp'] = pd.to_datetime(e['Timestamp'], format='%d.%m.%Y %H:%M')
    blockages = pd.to_datetime(np.sort(e.loc[e['Error'] == 'Extruder blocked', 'Timestamp'].values))
    p = d[d['State'] == 'printing'].copy()
    p['E'] = p['Energy'] * 1000                              # Wh per minute
    idx = p.index.values
    nxt = np.searchsorted(blockages.values, idx)
    mtb = np.where(nxt < len(blockages), (blockages.values[np.minimum(nxt, len(blockages) - 1)] - idx) / np.timedelta64(1, 'm'), np.inf)
    p['y'] = ((mtb > 0) & (mtb <= horizon)).astype(int)      # only used to EVALUATE here
    p['E30'] = p['E'].rolling(30).mean()
    p['E120'] = p['E'].rolling(120).mean()
    p['slope'] = p['E120'] - p['E120'].shift(240)
    return p.dropna(subset=FEATURES).iloc[::10], blockages

def evaluate_alarms(alarm, test, blockages):
    """alarm: boolean Series on the index of `test`. Counts blockages announced within 6 h and false alarm episodes per week."""
    t0, t1 = test.index[0], test.index[-1]
    blk = [t for t in blockages if t0 <= t <= t1]
    announced = sum(bool(alarm[(alarm.index >= t - pd.Timedelta(hours=6)) & (alarm.index <= t)].any()) for t in blk)
    times = alarm[alarm].index
    weeks = (t1 - t0).days / 7
    if len(times) == 0:
        return {'announced': f'0/{len(blk)}', 'false episodes / week': 0.0}
    new_episode = np.concatenate([[0], np.diff(times.values) > np.timedelta64(6, 'h')])   # a gap of more than 6 h starts a new episode
    ep = pd.DataFrame({'ep': np.cumsum(new_episode), 'y': test.loc[times, 'y'].values}).groupby('ep')['y'].max()
    return {'announced': f'{announced}/{len(blk)}', 'false episodes / week': round(float((ep == 0).sum() / weeks), 2)}

data, blockages = make_blockage_data(1)
cut = int(len(data) * 0.7)
reference, test = data.iloc[:cut], data.iloc[cut:]          # reference = 'what is normal', test = the future
print(len(reference), 'reference samples,', len(test), 'test samples,', len([t for t in blockages if t >= test.index[0]]), 'blockages in the test period')

## Step 1: What is normal?
Compute mean and standard deviation of `E120` in the **reference period** and standardise: `z = (E120 - mean) / std`. Plot the test period like a **control chart** (as in statistical process control) and mark the extruder blockages.

In [ ]:
mu, sd = reference['E120'].___(), reference['E120'].___()
z_ref = (reference['E120'] - mu) / sd
z_test = (test['E120'] - mu) / sd
print(f'normal E120: {mu:.3f} Wh/min, standard deviation {sd:.3f}')

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(z_test.index, z_test.values, '.', ms=1.5, color='#009EE0')
for t in [t for t in blockages if t >= test.index[0]]:
    ax.axvline(t, color='#E2001A', lw=1)
ax.axhline(3, color='#EE7F00', ls='--'); ax.set_ylabel('z of E120'); ax.set_title('test period; red lines = extruder blockages'); plt.show()

**What do you see before the red lines? Why does the curve not simply jump over the limit?**

*Your answer:*

## Step 2: Set the limit from the normal data
The limit is a **quantile of the reference data**: with the 95 % quantile, 5 % of the normal samples would be reported. Then count how many blockages are announced and how many **false alarm episodes per week** occur.

In [ ]:
limit = np.quantile(z_ref, ___)
alarm = z_test > limit
print(f'limit z > {limit:.2f}  (E120 > {mu + limit * sd:.3f} Wh/min)')
print(evaluate_alarms(alarm, test, blockages))

## Step 3: Move the limit
The limit controls the number of false alarms. Try several quantiles.

In [ ]:
rows = []
for q in [0.90, 0.95, 0.98, ___]:
    lim = np.quantile(z_ref, q)
    r = evaluate_alarms(z_test > lim, test, blockages)
    rows.append({'quantile': q, 'limit (z)': round(lim, 2), **r})
pd.DataFrame(rows).set_index('quantile')

**Which limit would you choose for a technician who can check one alarm per day? Which for a plant where a stop costs 10 hours?**

*Your answer:*

## Step 4: Compare with the classifier of exercise 2
Train the KNN of exercise 2 (k = 25, scaled features) **with labels** on the same reference period and evaluate its warnings in the same way.

In [ ]:
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=___)).fit(reference[FEATURES], reference['y'])
knn_alarm = pd.Series(knn.predict(test[FEATURES]) == 1, index=test.index)
rows = [
    {'method': 'z-value of E120 (no labels)', 'needs labels': 'no', 'AUC': roc_auc_score(test['y'], z_test), **evaluate_alarms(z_test > np.quantile(z_ref, 0.95), test, blockages)},
    {'method': 'KNN k = 25 (with labels)', 'needs labels': 'yes', 'AUC': roc_auc_score(test['y'], knn.predict_proba(test[FEATURES])[:, 1]), **evaluate_alarms(knn_alarm, test, blockages)},
]
pd.DataFrame(rows).set_index('method').round(3)

The ROC curves show the **ranking quality** of both scores over all thresholds (the AUC). They do not say how many false alarms a limit causes per week: use the table above for that.

In [ ]:
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(7, 5))
fpr, tpr, _ = roc_curve(test['y'], z_test)
plt.plot(fpr, tpr, label=f'z-value (AUC = {auc(fpr, tpr):.2f})')
fpr, tpr, _ = roc_curve(test['y'], knn.predict_proba(test[FEATURES])[:, 1])
plt.plot(fpr, tpr, label=f'KNN k = 25 (AUC = {auc(fpr, tpr):.2f})')
plt.plot([0, 1], [0, 1], 'r--', label='random guess')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curves of the two scores')
plt.legend()
plt.show()

**Does the label help here? When would you prefer the classifier, when the anomaly detector?**

*Your answer:*

## Step 5 (optional, if time): A more complex detector
**Isolation Forest** uses all three features and learns what is *easy to isolate* (unusual). Fit it on the reference data, set the limit again from the reference, and compare it with the simple z-value (AUC of the score and alarms).

In [ ]:
iso = IsolationForest(n_estimators=200, random_state=0).fit(reference[___])
s_ref = pd.Series(-iso.score_samples(reference[FEATURES]), index=reference.index)    # higher = more unusual
s_test = pd.Series(-iso.score_samples(test[FEATURES]), index=test.index)
lim_iso = np.quantile(s_ref, 0.95)

print('AUC z-value of E120 :', round(roc_auc_score(test['y'], z_test), 3), evaluate_alarms(z_test > np.quantile(z_ref, 0.95), test, blockages))
print('AUC Isolation Forest:', round(roc_auc_score(test['y'], s_test), 3), evaluate_alarms(s_test > lim_iso, test, blockages))

**Does the more complex Isolation Forest win? Why (not)?**

*Your answer:*

## Step 6 (optional, moves to week 3): A new printer
Apply the method to **printer 2** (a printer with 18 blockages) in two ways: (a) with the baseline of **printer 2** itself (its first 70 %), (b) with the baseline of **printer 1**. What is the difference?

In [ ]:
data2, blockages2 = make_blockage_data(___)
cut2 = int(len(data2) * 0.7)
ref2, test2 = data2.iloc[:cut2], data2.iloc[cut2:]
mu2, sd2 = ref2['E120'].mean(), ref2['E120'].std()
print(f'normal E120: printer 1 {mu:.3f} (sd {sd:.3f}), printer 2 {mu2:.3f} (sd {sd2:.3f})')

z_own = (test2['E120'] - mu2) / sd2
lim_own = np.quantile((ref2['E120'] - mu2) / sd2, 0.95)
z_foreign = (test2['E120'] - mu) / sd                      # baseline of printer 1
print('own baseline    :', round(roc_auc_score(test2['y'], z_own), 3), evaluate_alarms(z_own > lim_own, test2, blockages2))
print('baseline of P1  :', round(roc_auc_score(test2['y'], z_foreign), 3), evaluate_alarms(z_foreign > np.quantile(z_ref, 0.95), test2, blockages2))

**What does a printer need before the anomaly detector can be used on it? What does this mean for a fleet of 30 printers?**

*Your answer:*